# Adversarial Patch Experiments — Analysis Notebook

This notebook generates all paper figures from the experiment results in `./exp/`.

## Dataset & Experiment Results

The full experiment data (captures, results, and pre-generated figures) can be downloaded from Google Drive:

**[Download experiment data folder](https://drive.google.com/drive/folders/1S2WRuR5FrsbmX3LmrklyRY7QpPuwGo4y?usp=sharing)**

### Generated Figures
All figures are saved to `./exp/paper_figures_pastel/` as PDFs (and PNGs alongside).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from torchvision.models import Inception_V3_Weights

# Load ImageNet class labels from torchvision model metadata
imagenet_weights = Inception_V3_Weights.IMAGENET1K_V1
IMAGENET_CATEGORIES = imagenet_weights.meta['categories']

def get_class_names(indices):
    """Get class names from ImageNet indices using torchvision metadata"""
    return [IMAGENET_CATEGORIES[i] for i in indices]

# Base path for all experiments
BASE_PATH = r"./exp"

# Model definitions
V1_MODELS = ['v1_inception', 'v1_resnet', 'v1_vgg', 'v1_vit']  # Seen (trained on)
V2_MODELS = ['v2_convnext', 'v2_efficientnet', 'v2_mobilenet', 'v2_swin']  # Unseen
DINO_MODEL = 'dino_standalone'

# Approaches to analyze
APPROACHES = ['Baseline_Noise', 'CAPAA', 'Patch']

# =============================================================================
# PASTEL COLOR PALETTE
# =============================================================================
# Using soft pastel colors for all bar plots
PASTEL_COLORS = {
    'Baseline_Noise': '#B8D4E3',  # Pastel blue-gray
    'CAPAA': '#F4B8B8',           # Pastel red/pink
    'Patch': '#B8E3C0'            # Pastel green
}

# Display names for experiments (for bar plot labels)
EXP_DISPLAY_NAMES = {
    'cup': 'Cup',
    'paper_towel': 'Paper\nTowel',
    'purse': 'Purse',
    'jeep': 'Jeep',
    'water_jug': 'Watering\nCan'
}

# =============================================================================
# AUTO-DISCOVERY: find the correct experiment folders from disk
# =============================================================================
ATTACK_APPROACHES = {'CAPAA', 'Patch', 'Patch_rejuv'}

def find_experiment_folders(scene_name, base_path):
    """
    Scan a scene directory and return (main_folder, baseline_noise_folder).
    - main_folder: the latest folder that contains CAPAA/Patch predictions
    - baseline_noise_folder: the latest folder that contains only Baseline_Noise
      (when Baseline_Noise is missing from main_folder)
    """
    scene_path = os.path.join(base_path, scene_name)
    if not os.path.isdir(scene_path):
        return None, None

    main_folder = None
    baseline_noise_folder = None
    main_key = ''
    bn_key = ''

    for folder_name in sorted(os.listdir(scene_path)):
        folder_path = os.path.join(scene_path, folder_name)
        csv_path = os.path.join(folder_path, 'data', 'raw_predictions.csv')
        if not (os.path.isdir(folder_path) and os.path.exists(csv_path)):
            continue
        try:
            df = pd.read_csv(csv_path, usecols=['approach'])
            approaches = set(df['approach'].unique())
        except Exception:
            continue

        has_attack = bool(approaches & ATTACK_APPROACHES)
        has_bn = 'Baseline_Noise' in approaches

        if has_attack:
            # Keep the most recent folder (folders are named with timestamps)
            if folder_name > main_key:
                main_key = folder_name
                main_folder = f'{scene_name}/{folder_name}'
        elif has_bn:
            if folder_name > bn_key:
                bn_key = folder_name
                baseline_noise_folder = f'{scene_name}/{folder_name}'

    return main_folder, baseline_noise_folder


# Experiment configurations: forbidden class indices are domain knowledge kept here.
# main_folder and baseline_noise_folder are discovered automatically from disk.
SCENE_CONFIGS = {
    'cup': {
        'forbidden_indices': [504, 968, 899],
    },
    'paper_towel': {
        'forbidden_indices': [700, 999],
    },
    'purse': {
        'forbidden_indices': [636, 414, 748],
    },
    'jeep': {
        'forbidden_indices': [817, 705, 609, 586, 436, 627, 468, 621, 803, 407, 408, 751, 717, 866, 661, 864],
        'hardcoded_pixel': {'CAPAA': 15.3, 'Patch': 3.4},
        'hardcoded_pixel_se': {'CAPAA': 0.3, 'Patch': 0.1}
    },
    'water_jug': {
        'forbidden_indices': [899, 849, 505],
    }
}

# Build the full EXPERIMENTS dict with auto-discovered folder paths
EXPERIMENTS = {}
for scene_name, cfg in SCENE_CONFIGS.items():
    main_f, bn_f = find_experiment_folders(scene_name, BASE_PATH)
    EXPERIMENTS[scene_name] = {
        'main_folder': main_f,
        'baseline_noise_folder': bn_f,
        'forbidden_indices': cfg['forbidden_indices'],
        'forbidden_classes': get_class_names(cfg['forbidden_indices']),
    }
    if 'hardcoded_pixel' in cfg:
        EXPERIMENTS[scene_name]['hardcoded_pixel'] = cfg['hardcoded_pixel']
    if 'hardcoded_pixel_se' in cfg:
        EXPERIMENTS[scene_name]['hardcoded_pixel_se'] = cfg['hardcoded_pixel_se']

print(f"Experiments to analyze: {list(EXPERIMENTS.keys())}")
print(f"V1 Models (Seen): {V1_MODELS}")
print(f"V2 Models (Unseen): {V2_MODELS}")
print(f"\nAuto-discovered folders:")
for name, cfg in EXPERIMENTS.items():
    print(f"  {name}:")
    print(f"    main_folder          = {cfg['main_folder']}")
    print(f"    baseline_noise_folder= {cfg['baseline_noise_folder']}")
print(f"\nPastel Color Palette:")
for k, v in PASTEL_COLORS.items():
    print(f"  {k}: {v}")


In [ ]:
def load_experiment_data(exp_name, exp_config):
    """Load data for a single experiment, combining main and baseline_noise if needed"""
    base_path = exp_config.get('base_path', BASE_PATH)
    main_path = os.path.join(base_path, exp_config['main_folder'], 'data', 'raw_predictions.csv')
    
    if not os.path.exists(main_path):
        print(f"  Warning: {main_path} not found")
        return None
    
    df = pd.read_csv(main_path)
    print(f"  {exp_name}: Loaded {len(df)} predictions from main")
    
    if exp_config.get('baseline_noise_folder'):
        bn_base_path = exp_config.get('baseline_noise_base_path', base_path)
        bn_path = os.path.join(bn_base_path, exp_config['baseline_noise_folder'], 'data', 'raw_predictions.csv')
        if os.path.exists(bn_path):
            df_bn = pd.read_csv(bn_path)
            df_bn = df_bn[df_bn['approach'] == 'Baseline_Noise']
            df = pd.concat([df, df_bn], ignore_index=True)
            print(f"  {exp_name}: Added {len(df_bn)} baseline_noise predictions")
    
    forbidden = exp_config['forbidden_classes']
    df['success'] = ~df['predicted_class'].isin(forbidden)
    
    return df

def load_pixel_change_data(exp_name, exp_config):
    """Load pixel change analysis data for an experiment, including SE"""
    base_path = exp_config.get('base_path', BASE_PATH)
    main_path = os.path.join(base_path, exp_config['main_folder'], 'data', 'pixel_change_analysis.csv')
    
    pixel_data = {}
    pixel_se_data = {}
    
    if 'hardcoded_pixel' in exp_config:
        hardcoded_se = exp_config.get('hardcoded_pixel_se', {})
        for approach, val in exp_config['hardcoded_pixel'].items():
            pixel_data[approach] = val
            pixel_se_data[approach] = hardcoded_se.get(approach, 0.0)
        print(f"  {exp_name}: Using hardcoded pixel values: {exp_config['hardcoded_pixel']}")
    
    if os.path.exists(main_path):
        df = pd.read_csv(main_path)
        for approach in df['approach'].unique():
            approach_data = df[df['approach'] == approach]
            pixel_data[approach] = approach_data['pixel_change_pct'].mean()
            n = len(approach_data)
            if n > 1:
                pixel_se_data[approach] = approach_data['pixel_change_pct'].std(ddof=1) / np.sqrt(n)
            else:
                pixel_se_data[approach] = 0.0
        print(f"  {exp_name}: Loaded pixel change data ({len(df)} entries)")
    
    if exp_config.get('baseline_noise_folder'):
        bn_base_path = exp_config.get('baseline_noise_base_path', base_path)
        bn_path = os.path.join(bn_base_path, exp_config['baseline_noise_folder'], 'data', 'pixel_change_analysis.csv')
        if os.path.exists(bn_path):
            df_bn = pd.read_csv(bn_path)
            df_bn = df_bn[df_bn['approach'] == 'Baseline_Noise']
            if len(df_bn) > 0:
                pixel_data['Baseline_Noise'] = df_bn['pixel_change_pct'].mean()
                n = len(df_bn)
                if n > 1:
                    pixel_se_data['Baseline_Noise'] = df_bn['pixel_change_pct'].std(ddof=1) / np.sqrt(n)
                else:
                    pixel_se_data['Baseline_Noise'] = 0.0
                print(f"  {exp_name}: Added baseline_noise pixel data ({len(df_bn)} entries)")
    
    if 'Baseline' not in pixel_data:
        pixel_data['Baseline'] = 0.0
        pixel_se_data['Baseline'] = 0.0
    
    return (pixel_data, pixel_se_data) if pixel_data else (None, None)

# Load all experiments
print("Loading all experiments...")
all_data = {}
all_pixel_data = {}
all_pixel_se_data = {}
for exp_name, exp_config in EXPERIMENTS.items():
    df = load_experiment_data(exp_name, exp_config)
    if df is not None:
        all_data[exp_name] = df
        print(f"  {exp_name}: {df['approach'].unique()}")
    
    pixel_data, pixel_se_data = load_pixel_change_data(exp_name, exp_config)
    if pixel_data:
        all_pixel_data[exp_name] = pixel_data
        all_pixel_se_data[exp_name] = pixel_se_data

print(f"\nLoaded {len(all_data)} experiments successfully")
print(f"Pixel change data available for: {list(all_pixel_data.keys())}")

In [ ]:
def calc_se(data):
    """Calculate standard error of the mean"""
    if len(data) == 0:
        return np.nan
    return np.std(data, ddof=1) / np.sqrt(len(data)) * 100

def build_condensed_table(df, approaches, v1_models, v2_models, dino_model, pixel_data=None, pixel_se_data=None):
    """Build condensed table with individual models, group averages, standard errors, and pixel change"""
    results = []
    
    for approach in approaches:
        if approach not in df['approach'].unique():
            continue
        
        row = {'Method': approach}
        
        if pixel_data and approach in pixel_data:
            row['Pixel%'] = pixel_data[approach]
            row['Pixel_SE'] = pixel_se_data.get(approach, np.nan) if pixel_se_data else np.nan
        else:
            row['Pixel%'] = np.nan
            row['Pixel_SE'] = np.nan
        
        v1_rates = []
        for model in v1_models:
            model_data = df[(df['approach'] == approach) & (df['model'] == model)]
            rate = model_data['success'].mean() * 100 if len(model_data) > 0 else np.nan
            se = calc_se(model_data['success']) if len(model_data) > 0 else np.nan
            row[model.replace('v1_', 'Seen-')] = rate
            row[model.replace('v1_', 'Seen-') + '_SE'] = se
            if not np.isnan(rate):
                v1_rates.append(rate)
        row['Seen-Avg'] = np.mean(v1_rates) if v1_rates else np.nan
        row['Seen-Avg_SE'] = np.std(v1_rates) / np.sqrt(len(v1_rates)) if len(v1_rates) > 0 else np.nan
        
        dino_data = df[(df['approach'] == approach) & (df['model'] == dino_model)]
        row['DINO'] = dino_data['success'].mean() * 100 if len(dino_data) > 0 else np.nan
        row['DINO_SE'] = calc_se(dino_data['success']) if len(dino_data) > 0 else np.nan
        
        v2_rates = []
        for model in v2_models:
            model_data = df[(df['approach'] == approach) & (df['model'] == model)]
            rate = model_data['success'].mean() * 100 if len(model_data) > 0 else np.nan
            se = calc_se(model_data['success']) if len(model_data) > 0 else np.nan
            row[model.replace('v2_', 'Unseen-')] = rate
            row[model.replace('v2_', 'Unseen-') + '_SE'] = se
            if not np.isnan(rate):
                v2_rates.append(rate)
        row['Unseen-Avg'] = np.mean(v2_rates) if v2_rates else np.nan
        row['Unseen-Avg_SE'] = np.std(v2_rates) / np.sqrt(len(v2_rates)) if len(v2_rates) > 0 else np.nan
        
        results.append(row)
    
    table = pd.DataFrame(results)
    col_order = ['Method', 'Pixel%', 'Pixel_SE',
                 'Seen-inception', 'Seen-inception_SE', 'Seen-resnet', 'Seen-resnet_SE', 
                 'Seen-vgg', 'Seen-vgg_SE', 'Seen-vit', 'Seen-vit_SE', 'Seen-Avg', 'Seen-Avg_SE',
                 'DINO', 'DINO_SE',
                 'Unseen-convnext', 'Unseen-convnext_SE', 'Unseen-efficientnet', 'Unseen-efficientnet_SE', 
                 'Unseen-mobilenet', 'Unseen-mobilenet_SE', 'Unseen-swin', 'Unseen-swin_SE', 'Unseen-Avg', 'Unseen-Avg_SE']
    return table[[c for c in col_order if c in table.columns]]

# Build condensed tables for all experiments
all_tables = {}
for exp_name, df in all_data.items():
    approaches_to_use = ['CAPAA', 'Patch'] if EXPERIMENTS[exp_name].get('no_baseline') else APPROACHES
    pixel_data = all_pixel_data.get(exp_name, None)
    pixel_se_data = all_pixel_se_data.get(exp_name, None)
    all_tables[exp_name] = build_condensed_table(df, approaches_to_use, V1_MODELS, V2_MODELS, DINO_MODEL, pixel_data, pixel_se_data)
    print(f"\n{exp_name.upper()} (Forbidden: {EXPERIMENTS[exp_name]['forbidden_classes']})")
    display_cols = [c for c in all_tables[exp_name].columns if not c.endswith('_SE')]
    print(all_tables[exp_name][display_cols].to_string(index=False))

In [ ]:
# =============================================================================
# MAIN BAR CHART WITH PASTEL COLORS - All Methods Across All Setups
# =============================================================================

# Create output directory for paper figures
PAPER_FIGURES_PATH = os.path.join(BASE_PATH, 'paper_figures_pastel')
os.makedirs(PAPER_FIGURES_PATH, exist_ok=True)
print(f"Pastel figures will be saved to: {PAPER_FIGURES_PATH}")

# Grouped bar chart: All methods across all setups with pastel colors
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

# Metrics to compare
metrics = ['Pixel%', 'Seen-Avg', 'DINO', 'Unseen-Avg']
se_cols = ['Pixel_SE', 'Seen-Avg_SE', 'DINO_SE', 'Unseen-Avg_SE']
metric_titles = ['Pixel Change % (Lower = Better)', 'Seen-Avg Success Rate %', 'DINO Success Rate %', 'Unseen-Avg Success Rate %']

# Legend display names
legend_labels = {'Baseline_Noise': 'Baseline', 'CAPAA': 'CAPAA', 'Patch': '3PA'}

experiments_order = ['cup', 'paper_towel', 'purse', 'jeep', 'water_jug']
methods_order = ['Baseline_Noise', 'CAPAA', 'Patch']

for idx, (metric, se_col, title) in enumerate(zip(metrics, se_cols, metric_titles)):
    ax = axes[idx]
    
    # Gather data and SE for this metric
    data_by_method = {m: [] for m in methods_order}
    se_by_method = {m: [] for m in methods_order}
    exp_labels = []
    
    for exp_name in experiments_order:
        if exp_name not in all_tables:
            continue
        table = all_tables[exp_name]
        exp_labels.append(EXP_DISPLAY_NAMES.get(exp_name, exp_name.replace('_', '\n').title()))
        
        for method in methods_order:
            method_row = table[table['Method'] == method]
            if len(method_row) > 0 and metric in method_row.columns:
                val = method_row[metric].values[0]
                se = method_row[se_col].values[0] if se_col in method_row.columns else 0
                data_by_method[method].append(val if not pd.isna(val) else 0)
                se_by_method[method].append(se if not pd.isna(se) else 0)
            else:
                data_by_method[method].append(0)
                se_by_method[method].append(0)
    
    # Add "Average" column
    exp_labels.append('Average')
    for method in methods_order:
        method_vals = data_by_method[method]
        method_ses = se_by_method[method]
        avg_val = np.mean(method_vals) if method_vals else 0
        avg_se = np.sqrt(np.sum([s**2 for s in method_ses])) / len(method_ses) if method_ses else 0
        data_by_method[method].append(avg_val)
        se_by_method[method].append(avg_se)
    
    # Plot grouped bars with PASTEL colors
    x = np.arange(len(exp_labels))
    width = 0.25
    
    for i, method in enumerate(methods_order):
        offset = (i - 1) * width
        bars = ax.bar(x + offset, data_by_method[method], width, 
                     label=legend_labels[method], color=PASTEL_COLORS[method], alpha=0.95,
                     edgecolor='#555555', linewidth=0.8,
                     yerr=se_by_method[method], capsize=3, 
                     error_kw={'elinewidth': 1, 'capthick': 1, 'ecolor': '#555555'})
        
        # Add value labels
        for j, bar in enumerate(bars):
            height = bar.get_height()
            se = se_by_method[method][j]
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2, height + se + 1,
                       f'{height:.1f}', ha='center', va='bottom', fontsize=10, rotation=0)
    
    # Add vertical separator line before "Average"
    ax.axvline(x=len(exp_labels) - 1.5, color='gray', linewidth=2, linestyle='--', alpha=0.7)
    
    ax.set_xlabel('Scene', fontweight='bold', fontsize=14)
    ax.set_ylabel(title, fontweight='bold', fontsize=14)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(exp_labels, fontsize=11, fontweight='bold')
    ax.tick_params(axis='y', labelsize=12)
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if metric == 'Pixel%':
        ax.set_ylim(0, 70)
        ax.annotate('↓ Lower is better', xy=(0.02, 0.95), xycoords='axes fraction',
                   fontsize=11, color='#2E7D32', fontweight='bold')
    else:
        ax.set_ylim(0, 120)
        ax.annotate('↑ Higher is better', xy=(0.02, 0.95), xycoords='axes fraction',
                   fontsize=11, color='#2E7D32', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(BASE_PATH, 'all_methods_comparison_pastel.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(PAPER_FIGURES_PATH, 'all_methods_comparison_pastel.pdf'), format='pdf', bbox_inches='tight')
plt.show()

print(f"\nChart saved to: {os.path.join(BASE_PATH, 'all_methods_comparison_pastel.png')}")
print(f"PDF saved to: {os.path.join(PAPER_FIGURES_PATH, 'all_methods_comparison_pastel.pdf')}")

In [ ]:
# =============================================================================
# BAR PLOTS FOR EACH SCENARIO WITH PASTEL COLORS
# =============================================================================

legend_labels = {'Baseline_Noise': 'Baseline', 'CAPAA': 'CAPAA', 'Patch': '3PA'}

for exp_name, table in all_tables.items():
    # Get metrics
    metrics = ['Pixel%', 'Seen-inception', 'Seen-resnet', 'Seen-vgg', 'Seen-vit', 'Seen-Avg',
               'DINO', 'Unseen-convnext', 'Unseen-efficientnet', 'Unseen-mobilenet', 'Unseen-swin', 'Unseen-Avg']
    metrics = [m for m in metrics if m in table.columns]
    
    display_labels = [m.replace('Seen-', '').replace('Unseen-', '') for m in metrics]
    
    methods = table['Method'].tolist()
    n_methods = len(methods)
    n_metrics = len(metrics)
    
    fig, ax = plt.subplots(figsize=(18, 6))
    
    x = np.arange(n_metrics)
    width = 0.25
    
    for i, method in enumerate(methods):
        row = table[table['Method'] == method].iloc[0]
        values = []
        errors = []
        
        for metric in metrics:
            val = row.get(metric, np.nan)
            se_col = metric + '_SE' if metric != 'Pixel%' else 'Pixel_SE'
            se = row.get(se_col, 0) if se_col in row else 0
            
            values.append(val if not pd.isna(val) else 0)
            errors.append(se if not pd.isna(se) else 0)
        
        offset = (i - (n_methods - 1) / 2) * width
        bars = ax.bar(x + offset, values, width, 
                     label=legend_labels.get(method, method), 
                     color=PASTEL_COLORS.get(method, '#A8D8EA'), 
                     alpha=0.95, edgecolor='#555555', linewidth=0.8,
                     yerr=errors, capsize=3, 
                     error_kw={'elinewidth': 1, 'capthick': 1, 'ecolor': '#555555'})
        
        for j, bar in enumerate(bars):
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2, height + errors[j] + 2,
                       f'{height:.1f}', ha='center', va='bottom', fontsize=11, rotation=0)
    
    ax.set_xlabel('Metric / Classifier Model', fontweight='bold', fontsize=15)
    ax.set_ylabel('Success Rate (%) / Pixel Change (%)', fontweight='bold', fontsize=15)
    exp_display = EXP_DISPLAY_NAMES.get(exp_name, exp_name.replace('_', ' ').title()).replace('\n', ' ').upper()
    ax.set_title(f"{exp_display} - Attack Performance by Method (Pastel)", 
                 fontsize=18, fontweight='bold', pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=12)
    ax.legend(loc='upper right', fontsize=12)
    ax.set_ylim(0, 115)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Vertical separators
    ax.axvline(x=0.5, color='gray', linewidth=1, linestyle='--', alpha=0.5)
    ax.axvline(x=5.5, color='gray', linewidth=1, linestyle='--', alpha=0.5)
    ax.axvline(x=6.5, color='gray', linewidth=1, linestyle='--', alpha=0.5)
    
    # Group labels
    ax.text(0, 108, 'Pixel', ha='center', fontsize=12, fontweight='bold', color='#E57373')
    ax.text(3, 108, 'Seen', ha='center', fontsize=12, fontweight='bold', color='#64B5F6')
    ax.text(6, 108, 'DINO', ha='center', fontsize=12, fontweight='bold', color='#BA68C8')
    ax.text(9, 108, 'Unseen', ha='center', fontsize=12, fontweight='bold', color='#81C784')
    
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_PATH, f'bar_plot_{exp_name}_pastel.png'), dpi=150, bbox_inches='tight')
    plt.savefig(os.path.join(PAPER_FIGURES_PATH, f'bar_plot_{exp_name}_pastel.pdf'), format='pdf', bbox_inches='tight')
    plt.show()
    
    print(f"Saved: bar_plot_{exp_name}_pastel.png/pdf")


In [ ]:
# =============================================================================
# CAPAA VS PATCH COMPARISON WITH PASTEL COLORS
# =============================================================================

# Build comparison data
comparison_data = []
for exp_name, table in all_tables.items():
    capaa_row = table[table['Method'] == 'CAPAA']
    patch_row = table[table['Method'] == 'Patch']
    
    if len(capaa_row) > 0 and len(patch_row) > 0:
        capaa = capaa_row.iloc[0]
        patch = patch_row.iloc[0]
        
        comparison_data.append({
            'Experiment': exp_name,
            'CAPAA Pixel%': capaa.get('Pixel%', np.nan),
            'Patch Pixel%': patch.get('Pixel%', np.nan),
            'CAPAA Seen-Avg': capaa.get('Seen-Avg', np.nan),
            'Patch Seen-Avg': patch.get('Seen-Avg', np.nan),
            'CAPAA DINO': capaa.get('DINO', np.nan),
            'Patch DINO': patch.get('DINO', np.nan),
            'CAPAA Unseen-Avg': capaa.get('Unseen-Avg', np.nan),
            'Patch Unseen-Avg': patch.get('Unseen-Avg', np.nan),
        })

comparison_df = pd.DataFrame(comparison_data)

# Bar chart with pastel colors
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

groups = ['Seen-Avg', 'DINO', 'Unseen-Avg']
group_titles = ['Seen Models', 'DINO', 'Unseen Models']

# Pastel colors for CAPAA and Patch only
pastel_capaa = '#F4B8B8'  # Pastel pink/red
pastel_patch = '#B8E3C0'  # Pastel green

experiments = list(comparison_df['Experiment'])
x = np.arange(len(experiments))
width = 0.35

for idx, (group, title) in enumerate(zip(groups, group_titles)):
    ax = axes[idx]
    
    capaa_vals = comparison_df[f'CAPAA {group}'].values
    patch_vals = comparison_df[f'Patch {group}'].values
    
    bars1 = ax.bar(x - width/2, capaa_vals, width, label='CAPAA', 
                   color=pastel_capaa, alpha=0.95, edgecolor='#555555', linewidth=0.8)
    bars2 = ax.bar(x + width/2, patch_vals, width, label='3PA', 
                   color=pastel_patch, alpha=0.95, edgecolor='#555555', linewidth=0.8)
    
    ax.set_xlabel('Experiment', fontweight='bold', fontsize=14)
    ax.set_ylabel('Success Rate (%)', fontweight='bold', fontsize=14)
    ax.set_title(title, fontsize=17, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([EXP_DISPLAY_NAMES.get(e, e.replace('_', '\n').title()) for e in experiments], rotation=0, fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=12)
    ax.legend(fontsize=12)
    ax.set_ylim(0, 105)
    
    # Value labels
    for bar in bars1:
        if not np.isnan(bar.get_height()):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                   f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=12)
    for bar in bars2:
        if not np.isnan(bar.get_height()):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                   f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=12)

plt.suptitle('CAPAA vs 3PA - Attack Success Rate Comparison (Pastel)', fontsize=19, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_PATH, 'capaa_vs_patch_comparison_pastel.png'), dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(PAPER_FIGURES_PATH, 'capaa_vs_patch_comparison_pastel.pdf'), format='pdf', bbox_inches='tight')
plt.show()

print(f"Comparison chart saved!")

In [ ]:
# =============================================================================
# SUMMARY: Print pastel color codes for reference
# =============================================================================

print("\n" + "="*60)
print("PASTEL COLOR PALETTE USED IN THIS NOTEBOOK")
print("="*60)
print("\nMethod Colors:")
for method, color in PASTEL_COLORS.items():
    print(f"  {method:20s} -> {color}")

print("\nGroup Label Colors (softer variants):")
print(f"  Pixel  -> #E57373 (Pastel red)")
print(f"  Seen   -> #64B5F6 (Pastel blue)")
print(f"  DINO   -> #BA68C8 (Pastel purple)")
print(f"  Unseen -> #81C784 (Pastel green)")

print("\nAll figures saved to:")
print(f"  PNG: {BASE_PATH}")
print(f"  PDF: {PAPER_FIGURES_PATH}")